# Dataset 1 — Online Retail II EDA + RFM Classification

## Review 1 — Classification Part A

**Purpose:** Build a complete, explainable customer-level classification pipeline from transaction data.

**Rubric coverage in this notebook:**
- **A1 — Dataset loading & audit (1 mark):** shape, data types, missing values, duplicates and basic target/data checks.
- **A2 — EDA visualisations (2 marks):** feature distributions, correlation heatmap, target distribution and feature-target relationships.
- **A3 — Insight commentary (1 mark):** written observations accompany the major EDA sections.
- **B1 — Data cleaning (1 mark):** missing values, duplicates, transaction validity and outliers are checked with a justified treatment.
- **B2 — Encoding, scaling & splitting (1 mark):** leakage-safe preprocessing and stratified 80/20 split.
- **B3 — Feature engineering (1 mark):** RFM features and a justified classification target.
- **D1 — Classification implementation (2 marks):** all five Part-A algorithms are trained.
- **D2 — Classification evaluation (1 mark):** Accuracy, Precision, Recall, weighted F1, ROC-AUC and confusion matrices.

**Important leakage rule:** `Frequency` defines `Repeat_Customer`, so it is deliberately excluded from the predictors.

In [ ]:
from pathlib import Path
import datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

sns.set_theme(style="whitegrid")

class IQRClipper(BaseEstimator, TransformerMixin):
    """Cap numeric predictors using IQR bounds learned from fit data only."""
    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        q1 = X_df.quantile(0.25)
        q3 = X_df.quantile(0.75)
        iqr = q3 - q1
        self.lower_ = q1 - 1.5 * iqr
        self.upper_ = q3 + 1.5 * iqr
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        return X_df.clip(self.lower_, self.upper_, axis=1).to_numpy()

RANDOM_STATE = 42
N_JOBS = 2
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = Path("../data/online_retail_II.xlsx")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}. Put online_retail_II.xlsx in data/ and rerun."
    )

print("Loading Online Retail II dataset...")
df = pd.read_excel(DATA_PATH)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
print("Loaded:", DATA_PATH.resolve())
print("Shape:", df.shape)

## A1 — Dataset audit (1 mark)

We first report the dataset shape, data types, missing values, duplicate rows, and basic descriptive statistics. This makes data-quality issues visible before any modelling decision.

### A1 — Dataset loading and audit

**What this section demonstrates:** the first rubric item requires the dataset structure and quality to be reported before modelling. We therefore inspect shape, column data types, missing-value counts, duplicate rows and numerical summaries.

**Why it matters:** an ML model is only meaningful after we understand what each column represents and whether the raw data contains quality problems.

In [ ]:
print("\n--- Dataset Shape ---")
print(df.shape)

print("\n--- Data Types ---")
print(df.dtypes)

print("\n--- Missing Values ---")
missing_summary = df.isna().sum().sort_values(ascending=False)
print(missing_summary)

print("\n--- Exact Duplicate Rows ---")
print(df.duplicated().sum())

print("\n--- Numeric Summary ---")
display(df[['Quantity', 'Price']].describe().T)

print("\n--- Unique Counts ---")
for col in df.columns:
    print(f"{col:15s}: {df[col].nunique(dropna=True):,}")

### Audit observations

- `Customer ID` contains missing values, so customer-level behavioural features cannot be computed reliably for those rows.
- `Description` has some missing values but is not required for the final RFM model, so we still fill it to keep the cleaned transaction table internally consistent.
- Exact duplicate rows are explicitly checked and removed before customer-level aggregation.
- Transactional extremes are audited rather than blindly deleted because unusually large purchases can be genuine business observations.

### B1 — Data cleaning

**Cleaning strategy:**
1. Remove exact duplicate transaction rows because they repeat the same observation.
2. Remove rows without `Customer ID` because customer-level RFM cannot be computed for anonymous transactions.
3. Fill missing product descriptions with `UNKNOWN` so the cleaned transaction table has no null description values.
4. Audit negative quantities and non-positive prices before building purchase-based RFM features.

The treatment is deliberately justified rather than deleting observations without explanation.

In [ ]:
# Exact duplicate treatment
duplicate_count = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
print(f"Dropped {duplicate_count:,} exact duplicate rows.")

# Missing-value treatment
original_after_duplicates = df.shape[0]
df_clean = df.dropna(subset=['Customer ID', 'InvoiceDate']).copy()
df_clean['Description'] = df_clean['Description'].fillna('UNKNOWN')

print("\n--- After Missing-Value Cleaning ---")
print("Rows:", len(df_clean))
print(df_clean.isna().sum())

# Transaction validity audit
print("\n--- Transaction Sign Audit ---")
print("Negative Quantity rows:", int((df_clean['Quantity'] < 0).sum()))
print("Non-positive Price rows:", int((df_clean['Price'] <= 0).sum()))

## B1 — Outlier audit and treatment (part of the 1-mark data-cleaning criterion)

We first audit transaction-level outliers with the IQR rule. Because the final model works at customer level, we then audit the actual modelling predictors (`Recency` and `Monetary`) and apply IQR capping to the data used by the classifiers. The cap limits are learned from training data only, so the held-out test set is never used to define the treatment.


### B1 — Why we cap rather than delete

Removing every statistical outlier could discard genuine high-value customers. We therefore use **IQR capping (winsorisation)** for the final numeric predictors. Values outside the training-set IQR limits are capped at the corresponding lower/upper bound. This is an explicit treatment applied to the actual model inputs.


In [ ]:
def iqr_outlier_audit(df_in, columns):
    rows = []
    for col in columns:
        q1 = df_in[col].quantile(0.25)
        q3 = df_in[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        mask = (df_in[col] < lower) | (df_in[col] > upper)
        rows.append({
            'Feature': col,
            'Q1': q1,
            'Q3': q3,
            'IQR Lower Bound': lower,
            'IQR Upper Bound': upper,
            'Outlier Rows': int(mask.sum())
        })
    return pd.DataFrame(rows)

transaction_outlier_audit = iqr_outlier_audit(df_clean, ['Quantity', 'Price'])
print('--- Transaction-level outlier audit ---')
display(transaction_outlier_audit)
transaction_outlier_audit.to_csv(OUTPUT_DIR / 'retail_outlier_audit.csv', index=False)


## A2 — Raw-data EDA (2 marks)

The rubric asks for feature distributions, target distribution, a correlation heatmap, and feature-target relationships. The raw transaction table contains high-cardinality identifier/text fields, so meaningful distribution plots are used for numeric features and top-category frequency plots are used for categorical/text fields.

### A2 — Exploratory Data Analysis visualisations

The rubric asks for distributions, a correlation heatmap, a target distribution and feature-target relationships. Because the raw transaction table contains identifier/text columns with very high cardinality, distributions are shown for meaningful numeric variables and a country frequency plot is used for a major categorical field.

The correlation heatmap is descriptive only; the final classification features are constructed at customer level.

In [ ]:
# Numeric feature distributions
for col in ['Quantity', 'Price']:
    plt.figure(figsize=(9, 4.5))
    sns.histplot(df_clean[col].clip(lower=df_clean[col].quantile(0.01), upper=df_clean[col].quantile(0.99)), bins=50, kde=True, color='steelblue')
    plt.title(f'{col} Distribution (1st–99th Percentile View)')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'retail_{col.lower()}_distribution.png', dpi=180, bbox_inches='tight')
    plt.show()

# Country distribution
country_counts = df_clean['Country'].value_counts().head(15).sort_values()
plt.figure(figsize=(9, 6))
country_counts.plot(kind='barh')
plt.title('Top 15 Countries by Transaction Count')
plt.xlabel('Transactions')
plt.ylabel('Country')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'retail_country_distribution.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close('all')

# Correlation heatmap of substantive numeric transaction features
numeric_df = df_clean[['Quantity', 'Price']].copy()
plt.figure(figsize=(6, 4.5))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Heatmap — Transaction Numeric Features')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'retail_correlation_heatmap.png', dpi=180, bbox_inches='tight')
plt.show()

### Raw EDA observations

- Quantity and Price are strongly skewed, which is typical of transaction-level retail data.
- The raw `Quantity`/`Price` correlation is useful as a descriptive check, but these are not yet customer-level behavioural features.
- Country is highly imbalanced, with a small number of countries accounting for most transactions.
- High-cardinality fields such as Invoice, StockCode, and Description are not used directly for the customer-behaviour model.

### B3 — Feature engineering: RFM

We transform transaction-level records into one row per customer.

- **Recency:** number of days since the customer's last positive purchase.
- **Frequency:** number of distinct invoices.
- **Monetary:** total value of positive purchases.

A binary target, `Repeat_Customer`, is created from `Frequency > 1`.

**Leakage control:** because Frequency defines the target, it must not be used as an input feature.

In [ ]:
# Build RFM features from actual purchase transactions.
# Returns/negative quantities are excluded from RFM because the model is intended to describe purchasing behaviour.
purchases = df_clean[(df_clean['Quantity'] > 0) & (df_clean['Price'] > 0)].copy()
purchases['Total_Price'] = purchases['Quantity'] * purchases['Price']

latest_date = purchases['InvoiceDate'].max() + dt.timedelta(days=1)

rfm = purchases.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (latest_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('Total_Price', 'sum')
)

rfm_clean = rfm[rfm['Monetary'] > 0].copy()
rfm_clean['Repeat_Customer'] = (rfm_clean['Frequency'] > 1).astype(int)

print("\n--- RFM Dataset Shape ---")
print(rfm_clean.shape)
display(rfm_clean.head())

print("\n--- Class Distribution ---")
print(rfm_clean['Repeat_Customer'].value_counts().sort_index())

## B3 — Customer-level feature engineering (1 mark)

We convert transactions into one row per customer using **RFM**:

- **Recency:** days since the customer's most recent purchase.
- **Frequency:** number of unique purchase invoices.
- **Monetary:** total positive purchase value.
- **Repeat_Customer:** target = 1 when Frequency > 1.

`Frequency` is excluded from the predictors because it directly defines the target. The final model therefore uses **Recency and Monetary** only, avoiding target leakage.

### A2 — Customer-level EDA and target analysis

After RFM aggregation, the modelling population is customer-level rather than transaction-level. We now inspect the distributions of the final behavioural predictors, the class balance of `Repeat_Customer`, and the relationship between each predictor and the target.

This completes the rubric's required target and feature-target analysis.

In [ ]:
# Customer-level EDA: distributions and target
for col in ['Recency', 'Monetary']:
    upper = rfm_clean[col].quantile(0.99)
    plt.figure(figsize=(9, 4.5))
    sns.histplot(rfm_clean[col].clip(upper=upper), bins=40, kde=True, color='steelblue')
    plt.title(f'Customer-Level {col} Distribution (99th Percentile View)')
    plt.xlabel(col)
    plt.ylabel('Customers')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'rfm_{col.lower()}_distribution.png', dpi=180, bbox_inches='tight')
    plt.show()
    plt.close()

# Target distribution
plt.figure(figsize=(7, 4.5))
target_counts = rfm_clean['Repeat_Customer'].value_counts().sort_index()
sns.barplot(x=target_counts.index.astype(str), y=target_counts.values)
plt.title('Target Distribution — Repeat Customer')
plt.xlabel('Repeat_Customer (0 = No, 1 = Yes)')
plt.ylabel('Customers')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'classification_target_distribution.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close()

# EXACT rubric requirement: at least two scatter plots showing feature-target relationships.
rng = np.random.default_rng(RANDOM_STATE)
y_target = rfm_clean['Repeat_Customer'].to_numpy(dtype=float)
y_jitter = y_target + rng.uniform(-0.06, 0.06, size=len(y_target))

plt.figure(figsize=(9, 5))
sns.scatterplot(x=rfm_clean['Recency'], y=y_jitter, alpha=0.30, s=22)
plt.title('Scatter Plot — Recency vs Repeat Customer')
plt.xlabel('Recency (days)')
plt.ylabel('Repeat_Customer (jittered for visibility)')
plt.yticks([0, 1], ['No Repeat', 'Repeat'])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'classification_scatter_recency_target.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close()

monetary_log = np.log1p(rfm_clean['Monetary'])
plt.figure(figsize=(9, 5))
sns.scatterplot(x=monetary_log, y=y_jitter, alpha=0.30, s=22)
plt.title('Scatter Plot — Log-Monetary vs Repeat Customer')
plt.xlabel('log(1 + Monetary)')
plt.ylabel('Repeat_Customer (jittered for visibility)')
plt.yticks([0, 1], ['No Repeat', 'Repeat'])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'classification_scatter_monetary_target.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close()


### Customer-level EDA observations

- The target is based on repeat purchasing behaviour, so class balance must be preserved when splitting the data.
- The two required scatter plots show how Recency and Monetary vary across the two target classes.
- Monetary is highly skewed, so the second scatter plot uses `log(1 + Monetary)` for visual readability only; the actual model feature remains the original Monetary value.


### B2 — Train/test split and outlier-safe scaling

We use an 80/20 stratified split. The classifiers receive the original `Recency` and `Monetary` columns, but before each model sees them the `IQRClipper` learns caps from the training data and clips extreme predictor values. `StandardScaler` is then fitted on the treated training data. Both operations occur inside the pipeline, preventing leakage.


In [ ]:
# Final modelling data: Frequency is intentionally excluded to avoid direct target leakage.
X = rfm_clean[['Recency', 'Monetary']].copy()
y = rfm_clean['Repeat_Customer'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print('Training set shape:', X_train.shape)
print('Test set shape:', X_test.shape)
print('Training class proportions:')
print(y_train.value_counts(normalize=True).sort_index())
print('Test class proportions:')
print(y_test.value_counts(normalize=True).sort_index())

# Audit the actual model predictors on the training split before treatment.
model_outlier_audit = iqr_outlier_audit(X_train, ['Recency', 'Monetary'])
print('--- Model-feature outlier audit (training split) ---')
display(model_outlier_audit)
model_outlier_audit.to_csv(OUTPUT_DIR / 'classification_model_feature_outlier_audit.csv', index=False)


## B2 — Scaling, splitting and model setup (1 mark)

All five classifiers use the same predictors and held-out test set. The preprocessing pipeline is: **IQR clipping → StandardScaler → classifier**. The IQR capper is the explicit outlier-treatment step for the actual model inputs, and it is fitted separately inside each training/CV split.


### D1 — Classification implementation (all five Part-A algorithms)

The guideline requires five classifiers trained on the same preprocessed data. Each classifier uses the same leakage-safe pipeline: `IQRClipper` for outlier treatment, `StandardScaler` for scale normalization, and then the required estimator.


In [ ]:
classifiers_part_a = {
    'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE, max_iter=2000),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes (Gaussian)': GaussianNB(),
    'Decision Tree Classifier': DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=None),
    'Support Vector Machine (SVC)': SVC(probability=True, random_state=RANDOM_STATE, C=1.0, kernel='rbf')
}

results_list = []
predictions_by_model = {}
probabilities_by_model = {}
confusion_matrices = {}

for name, estimator in classifiers_part_a.items():
    model = Pipeline([
        ('outlier_treatment', IQRClipper()),
        ('scaler', StandardScaler()),
        ('model', estimator)
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    cm = confusion_matrix(y_test, predictions)

    results_list.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, predictions),
        'Precision': precision_score(y_test, predictions, average='weighted', zero_division=0),
        'Recall': recall_score(y_test, predictions, average='weighted', zero_division=0),
        'Weighted F1': f1_score(y_test, predictions, average='weighted', zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, probabilities)
    })
    predictions_by_model[name] = predictions
    probabilities_by_model[name] = probabilities
    confusion_matrices[name] = cm

    print(f'\n--- {name} ---')
    print('Confusion Matrix:')
    print(cm)
    print(classification_report(y_test, predictions, digits=4, zero_division=0))

comparison_table = (
    pd.DataFrame(results_list)
      .sort_values(by=['Accuracy', 'Weighted F1'], ascending=False)
      .reset_index(drop=True)
)

print('\n--- Complete Part-A Comparison Table ---')
display(comparison_table.style.format({
    'Accuracy':'{:.4f}', 'Precision':'{:.4f}', 'Recall':'{:.4f}',
    'Weighted F1':'{:.4f}', 'ROC-AUC':'{:.4f}'
}))
comparison_table.to_csv(OUTPUT_DIR / 'classification_part_a_results.csv', index=False)


## D2 — Confusion matrices and ROC curves (1 mark)

The confusion matrix shows the four prediction outcomes (TN, FP, FN, TP). ROC-AUC evaluates ranking quality across classification thresholds rather than using a single threshold only.

### D2 — Classification evaluation: confusion matrices and ROC-AUC

The rubric requires a confusion matrix and ROC-AUC in addition to the numerical metrics. The confusion matrix separates TN, FP, FN and TP. ROC-AUC measures how well the model ranks positive cases across classification thresholds.

In [ ]:
# Confusion matrices for all five models
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()
for idx, (name, cm) in enumerate(confusion_matrices.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[idx])
    axes[idx].set_title(name)
    axes[idx].set_xlabel('Predicted Label')
    axes[idx].set_ylabel('True Label')
axes[-1].axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'classification_confusion_matrices.png', dpi=180, bbox_inches='tight')
plt.show()

# ROC curves
plt.figure(figsize=(9, 6))
for name, probabilities in probabilities_by_model.items():
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    auc_value = roc_auc_score(y_test, probabilities)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc_value:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', linewidth=1, label='Random baseline')
plt.title('ROC Curves — Review 1 Part A')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'classification_roc_curves.png', dpi=180, bbox_inches='tight')
plt.show()

## General requirement — 5-fold cross-validation for the two strongest models

The general project guideline requires cross-validation for at least the two best-performing models. We identify the top two from the held-out test comparison, then use leakage-safe pipelines so scaling occurs separately inside each fold.

### General model-evaluation requirement — 5-fold cross-validation

The general guideline requires cross-validation for at least the two strongest models. The CV uses the complete scaling pipeline rather than a scaler fitted once on all training data, preventing fold-to-fold leakage.

In [ ]:
top_two_names = comparison_table.head(2)['Model'].tolist()
cv_rows = []

for name in top_two_names:
    estimator = classifiers_part_a[name]
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', estimator)
    ])

    cv = cross_validate(
        pipeline, X_train, y_train, cv=5,
        scoring={
            'accuracy':'accuracy',
            'precision':'precision_weighted',
            'recall':'recall_weighted',
            'f1':'f1_weighted',
            'roc_auc':'roc_auc'
        },
        n_jobs=N_JOBS, return_train_score=False
    )

    cv_rows.append({
        'Model': name,
        'CV Accuracy Mean': cv['test_accuracy'].mean(),
        'CV Precision Mean': cv['test_precision'].mean(),
        'CV Recall Mean': cv['test_recall'].mean(),
        'CV Weighted F1 Mean': cv['test_f1'].mean(),
        'CV ROC-AUC Mean': cv['test_roc_auc'].mean(),
        'CV ROC-AUC Std': cv['test_roc_auc'].std()
    })

cv_top2 = pd.DataFrame(cv_rows)
print('Top two baseline models for 5-fold CV:', top_two_names)
display(cv_top2.style.format('{:.4f}', subset=[c for c in cv_top2.columns if c != 'Model']))
cv_top2.to_csv(OUTPUT_DIR / 'classification_cv_top2.csv', index=False)

## Review-1 final interpretation and viva preparation

The final classification table reports Accuracy, Precision, Recall, Weighted F1 and ROC-AUC for all five required Part-A models. Confusion matrices and ROC curves provide additional diagnostic evidence.

The modelling pipeline also demonstrates two important quality controls: **IQR outlier treatment is applied to the actual predictors before each classifier**, and Frequency is excluded because it defines the target.


### Review-1 checklist / viva readiness

This final section prints the sizes of the raw transaction data, cleaned data, RFM customer table and train/test split. During the viva, be prepared to explain **why the data becomes customer-level, why Frequency is excluded, why stratification is used, and what each evaluation metric means**.

In [ ]:
# Compact reproducibility summary
print("\n=== FINAL DATA / MODEL SUMMARY ===")
print("Raw dataset rows:", len(df))
print("Clean transaction rows:", len(df_clean))
print("Purchase rows used for RFM:", len(purchases))
print("Customer-level rows:", len(rfm_clean))
print("Predictors used:", list(X.columns))
print("Target:", y.name)
print("Train/Test:", X_train.shape[0], "/", X_test.shape[0])
print("Top two for 5-fold CV:", top_two_names)
print("Saved outputs in:", OUTPUT_DIR.resolve())